# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset on predictors of adoption of indigenous and modern knowledge in rangeland management in Northern Kenya, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'
ds = mlc.Dataset(croissant_url)

# Access and print dataset name and description (use attributes, not dict subscripting)
print(f"{ds.metadata.name}: {ds.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs (all entities referenced by their `@id`).

In [ ]:
# List available record sets and their field @ids
print("Available Record Sets (@id):")
record_sets = [r for r in ds.record_sets]
for rset in record_sets:
    print(f"- {rset['@id']} (name: {rset.get('name', '')})")
    if 'field' in rset:
        print("    Fields:")
        for fld in rset['field']:
            print(f"      - {fld['@id']} (name: {fld.get('name', '')}, type: {fld.get('dataType', '')})")
if not record_sets:
    print("No record sets defined in metadata. Attempting to list records via available distributions...")
    dist_ids = getattr(ds.metadata, 'distribution', [])
    print(f"Distributions (@id): {dist_ids}")

## 3. Data Extraction
Load data from each discovered record set, referencing by their `@id`. If no explicit record sets exist, attempt to access records from main distribution(s).

In [ ]:
# If record sets were listed, use them; otherwise, use the first available distribution as fallback
dataframes = {}

if record_sets:
    # Extract all available record sets
    record_set_ids = [rset['@id'] for rset in record_sets]
    print("Extracting data from record sets:", record_set_ids)
    for record_set_id in record_set_ids:
        try:
            records = list(ds.records(record_set=record_set_id))
            if records:
                dataframes[record_set_id] = pd.DataFrame(records)
                print(f"Loaded {len(records)} records for record set {record_set_id}.")
            else:
                print(f"No records found for record set {record_set_id}.")
        except Exception as e:
            print(f"Could not load records for record set {record_set_id}: {e}")
else:
    # Fallback: try distributions if record sets are missing
    dist_ids = getattr(ds.metadata, 'distribution', [])
    if dist_ids:
        first_dist_id = dist_ids[0]['@id'] if isinstance(dist_ids[0], dict) and '@id' in dist_ids[0] else dist_ids[0]
        print(f"Attempting to read records from distribution {first_dist_id}...")
        try:
            records = list(ds.records(file_object=first_dist_id))
            if records:
                dataframes[first_dist_id] = pd.DataFrame(records)
                print(f"Loaded {len(records)} records from distribution {first_dist_id}.")
            else:
                print(f"No records found for distribution {first_dist_id}.")
        except Exception as e:
            print(f"Could not load records from distribution {first_dist_id}: {e}")
    else:
        print("No usable record sets or distributions found.")

# Display columns of the first (or only) loaded DataFrame, if available
if dataframes:
    first_df_key = list(dataframes.keys())[0]
    print(f"Columns in {first_df_key}:")
    print(dataframes[first_df_key].columns.tolist())
    dataframes[first_df_key].head()

## 4. Exploratory Data Analysis (EDA)
Apply filtering, normalization, and group analysis referencing all fields by their `@id` from the previous extraction.

In [ ]:
import numpy as np
# For demonstration, pick the first loaded DataFrame, and use the first numeric column (if any)
if dataframes:
    df = dataframes[first_df_key]
    # Identify numeric columns by dtype or content
    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not numeric_cols:
        # Try to infer numeric fields
        candidates = [c for c in df.columns if any(x in c.lower() for x in ['log', 'coef', 'standard', 'pvalue', 'iteration', 'value'])]
        numeric_cols = candidates
    if numeric_cols:
        numeric_field = numeric_cols[0]  # Use the first numeric/likely-numeric field
        print(f"Using numeric field for EDA: {numeric_field}")
        # Remove outliers (e.g., values greater than 2 std devs above mean)
        x = pd.to_numeric(df[numeric_field], errors='coerce')
        m = x.mean()
        s = x.std()
        threshold = m + 2 * s
        filtered_df = df[x <= threshold].copy()
        print(f"Filtered records with {numeric_field} <= {threshold:.2f} (mean+2std): {filtered_df.shape[0]}")
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (x - m) / s
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Group by a categorical column if available
        group_col_candidates = [c for c in df.columns if c != numeric_field and df[c].nunique() < 20 and df[c].dtype=='object']
        if group_col_candidates:
            group_field = group_col_candidates[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No appropriate group field found for grouping.")
    else:
        print("No numeric fields identified in the data for EDA.")
else:
    print("No data loaded; cannot perform EDA.")

## 5. Visualization
Visualize data distributions, referencing all fields by their `@id` inside code.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_cols:
    fig, ax = plt.subplots(figsize=(7,4))
    sns.histplot(pd.to_numeric(df[numeric_field], errors='coerce').dropna(), bins=25, kde=True, ax=ax)
    ax.set_title(f"Distribution of {numeric_field}")
    ax.set_xlabel(numeric_field)
    plt.show()
    # If grouped field exists, plot group-wise means
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(7,4))
        grouped_df.plot(kind='bar', ax=plt.gca())
        plt.ylabel(f"Mean {numeric_field}")
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric fields to visualize.")

## 6. Conclusion
In this notebook, we demonstrated how to programmatically load and inspect a Croissant-compliant FAIR data package using `mlcroissant`, referencing all record sets and fields by their `@id`. This process included metadata inspection, record set discovery, DataFrame creation, simple EDA, and basic visualization. Further domain-specific analysis can be performed using the DataFrame(s) extracted with this workflow.